In [ ]:
! head {WORK_DIR}/FOREST_PLOTS/combined_forest_data.csv

rsID,SNP(hg38),Cohort,BETA,SE,P,OR,L95,U95
rs144814361,chr10:119651405:C:T,Female [ALL],0.3569,0.04992457497847578,8.755e-13,1.428892973411833,1.2956957804167228,1.5757828035907038
rs144814361,chr10:119651405:C:T,Male [ALL],0.3431,0.0421172326479952,3.752e-16,1.4093096858487222,1.297643885095164,1.5305846337659639
rs144814361,chr10:119651405:C:T,Female [EUR],0.3569,0.04992457497847578,8.755e-13,1.428892973411833,1.2956957804167228,1.5757828035907038
rs144814361,chr10:119651405:C:T,Male [EUR],0.3431,0.0421172326479952,3.752e-16,1.4093096858487222,1.297643885095164,1.5305846337659639
rs144814361,chr10:119651405:C:T,Female [Biobank],0.3495,0.07719820352507023,5.974e-06,1.418358192172844,1.2191958731864252,1.6500547660533347
rs144814361,chr10:119651405:C:T,Male [Biobank],0.3923,0.06480022227966904,1.413e-09,1.4803817593195663,1.3038110651674941,1.6808648215027682
rs144814361,chr10:119651405:C:T,Female [CC],0.3622,0.06545121074389935,3.132e-08,1.4364862104746379,1.2635378791553893,1.6331070

In [ ]:
! grep "rs" {WORK_DIR}/FOREST_PLOTS/combined_forest_data.csv

rs3213916,chr14:87949673:G:A,Female [ALL],0.081,0.014177803225542573,1.109e-08,1.0843708965667602,1.0546526865627175,1.1149265121139709
rs3213916,chr14:87949673:G:A,Male [ALL],0.0224,0.0124049322926321,0.07096,1.0226527637746343,0.998088162600241,1.0478219404298998
rs3213916,chr14:87949673:G:A,Female [EUR],0.0749,0.014385501999121399,1.923e-07,1.0777763678587842,1.0478122468520004,1.1085973680921726
rs3213916,chr14:87949673:G:A,Male [EUR],0.0203,0.012662515408816436,0.1089,1.0205074463424153,0.9954916629977714,1.0461518531498233
rs3213916,chr14:87949673:G:A,Female [AJ],0.2854,0.08333921804681346,0.0006158,1.3302940393278702,1.1298163898102567,1.566344980504721
rs3213916,chr14:87949673:G:A,Male [AJ],0.0737,0.06243098557433007,0.2378,1.0764838119060318,0.9525004193435457,1.2166056557690414
rs3213916,chr14:87949673:G:A,Female [Biobank],0.0778,0.023411587570870942,0.0008901,1.0809064557593608,1.0324279779016232,1.1316612791498686
rs3213916,chr14:87949673:G:A,Male [Biobank],-0.0068,0.020507

In [ ]:
#!/usr/bin/env python3
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Input/output paths
infile = "../combined_forest_data.csv"
outdir = "../python_pngs"
os.makedirs(outdir, exist_ok=True)

# Load data
df = pd.read_csv(infile)
df["Cohort"] = df["Cohort"].astype(str).str.strip()

# Define the only valid cohorts, in the correct order
order = [
    "Female [ALL]",
    "Male [ALL]",
    "Female [EUR]",
    "Male [EUR]",
    "Female [Biobank]",
    "Male [Biobank]",
    "Female [CC]",
    "Male [CC]",
]

# Keep only rows in that list
df = df[df["Cohort"].isin(order)].copy()

# Group by SNP(hg38)
for snp, g in df.groupby("SNP(hg38)"):
    g = g.copy()
    g["Cohort"] = pd.Categorical(g["Cohort"], categories=order, ordered=True)
    g = g.sort_values("Cohort").reset_index(drop=True)

    # Grab rsID (should be unique within SNP group)
    rsid = g["rsID"].iloc[0]

    # Y positions
    y = np.arange(len(g))

    # Autoscale x with padding
    xmin = float(g["L95"].min()) * 0.9
    xmax = float(g["U95"].max()) * 1.1

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.set_xlim(xmin, xmax)

    # Plot CIs and ORs
    ax.errorbar(
        g["OR"], y,
        xerr=[g["OR"] - g["L95"], g["U95"] - g["OR"]],
        fmt="s", color="black", ecolor="black", elinewidth=1, capsize=3
    )

    # Reference line at OR = 1
    ax.axvline(1, color="grey", linestyle="--", linewidth=1)

    # Y labels
    ax.set_yticks(y)
    ax.set_yticklabels(g["Cohort"])
    ax.invert_yaxis()

    # Right-side text block
    right = ax.get_xlim()[1]
    x_text = right * 1.05

    # Add row values
    for yi, (OR, L95, U95, p) in enumerate(zip(g["OR"], g["L95"], g["U95"], g["P"])):
        label = f"{OR:.2f} [{L95:.2f}, {U95:.2f}]; P={p:.2E}"
        if p < 5e-8:
            label += " *"
        ax.text(x_text, yi, label, va="center", ha="left", fontsize=9, color="black")

    ax.set_xlabel("Odds Ratio (95% CI)")
    ax.set_title(f"SNP: {snp} | rsID: {rsid}")

    plt.tight_layout()

    # Replace ":" with "_" in filenames for safety
    safe_snp = snp.replace(":", "_")
    outfile = os.path.join(outdir, f"{safe_snp}_{rsid}_forest.png")

    plt.savefig(outfile, dpi=300, bbox_inches="tight")
    plt.close()

print(f"Forest plots saved in {outdir}")